# 01. RAG 개념 이해

## RAG란?

**RAG(Retrieval-Augmented Generation)** 는 LLM의 한계를 극복하기 위해  
**외부 문서를 검색(Retrieval)** 하고, 그 결과를 바탕으로 **답변을 생성(Generation)** 하는 기술임

---

## LLM의 한계

| 문제 | 설명 |
|---|---|
| 지식 단절 | 학습 데이터 이후의 최신 정보를 모름 |
| 할루시네이션 | 사실과 다른 내용을 그럴듯하게 생성 |
| 출처 불명 | 어디서 얻은 정보인지 알 수 없음 |
| 컨텍스트 제한 | 모든 문서를 프롬프트에 넣을 수 없음 |

---

## RAG 파이프라인

```
【 인덱싱 단계 (오프라인) 】

  문서 로딩  →  텍스트 분할  →  임베딩  →  벡터 스토어 저장

【 검색·생성 단계 (온라인) 】

  사용자 질문  →  임베딩  →  유사 문서 검색  →  LLM 답변 생성
```

---



## Naive RAG 구성 요소

| 단계 | 역할 | 도구 |
|---|---|---|
| Document Loader | 문서 불러오기 | TextLoader, PyPDFLoader |
| Text Splitter | 문서를 청크로 분할 | RecursiveCharacterTextSplitter |
| Embedding | 텍스트 → 벡터 변환 | OpenAIEmbeddings |
| Vector Store | 벡터 저장 및 검색 | FAISS, Chroma |
| Retriever | 관련 문서 검색 | as_retriever() |
| LLM | 최종 답변 생성 | ChatOpenAI |

## RAG vs 일반 LLM 비교 실습

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.getenv("LLM_BASE_URL", "https://monogpt.kr/api/monorouter/v1"), use_responses_api=False, model="gpt-4.1", temperature=0)
if not os.getenv("LLM_API_KEY"):
    raise ValueError("프로젝트 .env에 LLM_API_KEY를 설정하세요.")


In [2]:
# LLM 단독 호출 - 모델이 알고 있는 내용만 답변
question = "LangGraph는 무엇이고 어떤 개념을 사용하나요?"

response = llm.invoke(question)
print("[LLM 단독 답변]")
print(response.content)

[LLM 단독 답변]
**LangGraph**는 LLM(대형 언어 모델) 기반 애플리케이션을 위한 **상태 기반 멀티에이전트 워크플로우**를 설계하고 실행할 수 있게 해주는 오픈소스 프레임워크입니다. LangChain 팀에서 개발했으며, 복잡한 대화 흐름이나 여러 에이전트가 상호작용하는 시스템을 쉽게 구축할 수 있도록 돕습니다.

## 주요 개념

### 1. **그래프(Graph) 기반 워크플로우**
- LangGraph는 워크플로우를 **노드(상태)**와 **에지(전이)**로 구성된 그래프로 모델링합니다.
- 각 노드는 특정 작업(예: LLM 호출, 함수 실행 등)을 담당하고, 에지는 상태 간의 전환(다음에 어떤 노드로 이동할지 결정)을 정의합니다.
- 이를 통해 복잡한 분기, 반복, 조건부 실행 등 다양한 흐름을 유연하게 설계할 수 있습니다.

### 2. **상태(State)**
- 각 노드는 상태를 나타내며, 상태에는 대화의 맥락, 변수, 이전 결과 등이 저장될 수 있습니다.
- 상태를 기반으로 다음에 어떤 작업을 할지 결정할 수 있습니다.

### 3. **멀티에이전트(Multi-Agent)**
- 여러 LLM 에이전트가 서로 협력하거나 경쟁하는 구조를 쉽게 만들 수 있습니다.
- 예를 들어, 하나의 에이전트가 질문을 생성하고, 다른 에이전트가 답변을 검증하는 식의 협업이 가능합니다.

### 4. **LangChain과의 통합**
- LangGraph는 LangChain의 체인, 에이전트, 툴 등과 자연스럽게 연동됩니다.
- 기존 LangChain 생태계의 다양한 컴포넌트를 그래프 노드로 활용할 수 있습니다.

### 5. **비동기/동기 실행**
- 각 노드는 동기(synchronous) 또는 비동기(asynchronous)로 실행할 수 있어, 효율적인 워크플로우 설계가 가능합니다.

---

## 요약
**LangGraph**는 LLM 기반 애플리케이션에서 복잡한 상태 전이, 멀티에이전트 상호작용, 조건부 분기 등을 그래프 형태로 직관적이고 유

In [3]:
# 외부 문서를 직접 제공하면 더 정확한 답변 가능
context = """
LangGraph는 LangChain의 확장으로, 그래프 기반의 멀티 에이전트 워크플로우를 구성할 수 있음
State, Node, Edge 개념을 사용해 복잡한 AI 파이프라인을 구현함
"""

prompt = f"다음 문서를 참고해서 질문에 답하세요.\n\n문서: {context}\n\n질문: {question}"

response = llm.invoke(prompt)
print("[문서 기반 답변]")
print(response.content)

[문서 기반 답변]
LangGraph는 LangChain의 확장으로, 그래프 기반의 멀티 에이전트 워크플로우를 구성할 수 있는 도구입니다. LangGraph는 복잡한 AI 파이프라인을 구현하기 위해 State, Node, Edge라는 개념을 사용합니다.


---

## 정리

- RAG는 **외부 문서 → 검색 → 생성** 3단계로 동작함
- 다음 노트북부터 각 단계를 하나씩 실습함

    ```
    02_document_loader   → 문서 불러오기
    03_text_splitter     → 텍스트 분할
    04_embedding         → 임베딩
    05_vector_store      → 벡터 스토어
    06_retriever         → 검색
    07_rag_pipeline      → 전체 파이프라인
    ```